In [18]:
from _provenance import Sello
sello = Sello('00_test_datasets.ipynb')

In [23]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np
from sklearn.preprocessing import PowerTransformer

In [9]:
ruta='../data/prueba/intact_yj_max_wnd.csv'

In [10]:
df = pd.read_csv(ruta)
FEATURES = ['VAR_STD','Range','Energy','Peak_to_Peak','VAR','Max','MNF','SSC']
META = ['origin_db','subject','stimulus','window']
GROUPS = ['origin_db','subject','stimulus']
N_WINDOWS = 25 

In [11]:
df.head()

,origin_db,subject,stimulus,window,Channel 1_VAR_STD,Channel 2_VAR_STD,Channel 3_VAR_STD,Channel 4_VAR_STD,Channel 5_VAR_STD,Channel 6_VAR_STD,...,Channel 7_MNF,Channel 8_MNF,Channel 1_SSC,Channel 2_SSC,Channel 3_SSC,Channel 4_SSC,Channel 5_SSC,Channel 6_SSC,Channel 7_SSC,Channel 8_SSC
0,NinaProDB1,1,22,0,-1.183471,-1.124746,-1.324195,-1.063065,-1.579337,-1.180050,...,1.711272,0.202891,-1.842317,-1.791773,-1.522857,-1.561058,-1.625470,-1.547005,-1.596323,-1.319317
1,NinaProDB1,1,22,1,-1.186780,-1.124753,-1.323859,-1.057477,-1.242132,-1.177969,...,1.605942,-0.008948,-1.663923,-1.791773,-1.737845,-1.561058,-1.625470,-1.381797,-1.596323,-1.485911
2,NinaProDB1,1,22,2,-1.188979,-1.122673,-1.315849,-1.006803,-0.751191,-1.173322,...,1.622314,0.424317,-1.663923,-1.791773,-1.737845,-0.971757,-1.170485,-1.547005,-1.936079,-2.161486
3,NinaProDB1,1,22,3,-1.105428,0.703104,-1.132414,-0.703941,-0.141090,-1.107405,...,1.437350,0.922165,-1.663923,-1.074598,-1.297987,-0.248521,-0.932148,-1.381797,-1.936079,-1.821798
4,NinaProDB1,1,22,4,-0.810961,1.210666,-0.948888,-0.391821,0.172290,-0.776175,...,1.555775,1.644502,-1.479850,-0.010143,-0.314823,0.262037,0.084656,-1.045129,-0.942826,-1.485911


In [12]:
signal_cols=df.columns.difference(META).tolist()
signal_cols

['Channel 1_Energy',
 'Channel 1_MNF',
 'Channel 1_Max',
 'Channel 1_Peak_to_Peak',
 'Channel 1_Range',
 'Channel 1_SSC',
 'Channel 1_VAR',
 'Channel 1_VAR_STD',
 'Channel 2_Energy',
 'Channel 2_MNF',
 'Channel 2_Max',
 'Channel 2_Peak_to_Peak',
 'Channel 2_Range',
 'Channel 2_SSC',
 'Channel 2_VAR',
 'Channel 2_VAR_STD',
 'Channel 3_Energy',
 'Channel 3_MNF',
 'Channel 3_Max',
 'Channel 3_Peak_to_Peak',
 'Channel 3_Range',
 'Channel 3_SSC',
 'Channel 3_VAR',
 'Channel 3_VAR_STD',
 'Channel 4_Energy',
 'Channel 4_MNF',
 'Channel 4_Max',
 'Channel 4_Peak_to_Peak',
 'Channel 4_Range',
 'Channel 4_SSC',
 'Channel 4_VAR',
 'Channel 4_VAR_STD',
 'Channel 5_Energy',
 'Channel 5_MNF',
 'Channel 5_Max',
 'Channel 5_Peak_to_Peak',
 'Channel 5_Range',
 'Channel 5_SSC',
 'Channel 5_VAR',
 'Channel 5_VAR_STD',
 'Channel 6_Energy',
 'Channel 6_MNF',
 'Channel 6_Max',
 'Channel 6_Peak_to_Peak',
 'Channel 6_Range',
 'Channel 6_SSC',
 'Channel 6_VAR',
 'Channel 6_VAR_STD',
 'Channel 7_Energy',
 'Chann

In [14]:
def pca_fuse(df, features, n_channels=8, prefix=''):
    """PCA 1 componente por feature (covarianza, sin escalar).
    Devuelve df con META + una columna por feature."""
    out = df[META].copy()
    for feat in features:
        cols = [f'Channel {ch}_{feat}' for ch in range(1, n_channels+1)]
        cols = [c for c in cols if c in df.columns]
        X = df[cols].values.astype(float)
        mu = np.nanmean(X, axis=0); mu = np.where(np.isfinite(mu), mu, 0.0)
        Xc = np.where(np.isnan(X), mu, X)
        pc1 = PCA(n_components=1).fit_transform(Xc).ravel()
        ref = np.nanmean(Xc, axis=1)
        if np.corrcoef(pc1, ref)[0, 1] < 0: pc1 = -pc1
        out[feat] = pc1
        print(f'{feat:>12}: varianza explicada = '
              f'{PCA(n_components=1).fit(Xc).explained_variance_ratio_[0]:.3f}')
    return out

In [15]:
df_pca = pca_fuse(df, FEATURES)
df_pca.head()

     VAR_STD: varianza explicada = 0.584
       Range: varianza explicada = 0.502
      Energy: varianza explicada = 0.642
Peak_to_Peak: varianza explicada = 0.616
         VAR: varianza explicada = 0.485
         Max: varianza explicada = 0.653
         MNF: varianza explicada = 0.520
         SSC: varianza explicada = 0.603


,origin_db,subject,stimulus,window,VAR_STD,Range,Energy,Peak_to_Peak,VAR,Max,MNF,SSC
0,NinaProDB1,1,22,0,-3.674658,-1.322528,-3.721729,-5.089961,-1.726297,-3.826192,0.323148,-4.509600
1,NinaProDB1,1,22,1,-3.554762,-0.563412,-3.459465,-4.839646,-0.887406,-3.500691,0.904601,-4.526623
2,NinaProDB1,1,22,2,-3.345881,0.567020,-2.987383,-4.250516,0.401268,-2.824074,1.771488,-4.516673
3,NinaProDB1,1,22,3,-2.026584,1.543641,-2.190629,-1.976384,1.445895,-1.933262,2.559870,-3.562982
4,NinaProDB1,1,22,4,-0.756450,1.948758,-1.271010,-0.227537,1.860489,-1.034803,3.317405,-1.600617


In [38]:
df_pca.to_csv('../data/processed/df_pca_wnd_mean.csv', index=False)

In [39]:
def flatten_windows(df_pca, features, window_range=None, n_windows=25):
    """
    Aplana las ventanas de cada (origin_db, subject, stimulus).
    Orden feature-major: feature0 × w0..wn, feature1 × w0..wn, ...

    Parámetros:
        window_range : tupla (inicio, fin) — filtra por VALOR de columna 'window'
                       ej. (19, 25) → ventanas 19..25 (incluidas, 7 ventanas)
                           (0, 10)  → ventanas 0..10 (incluidas, 11 ventanas)
                       Si es None, usa las últimas n_windows ventanas (comportamiento original).
    """
    parts, keys = [], []
    g_sorted = df_pca.sort_values('window')

    for key, g in g_sorted.groupby(GROUPS, sort=True):
        if window_range is not None:
            start, end = window_range
            g = g[(g['window'] >= start) & (g['window'] <= end)]   # rango por valor
        else:
            g = g.tail(n_windows)                                  # últimas n

        if g.empty:
            continue

        arr = g[features].to_numpy()             # (n_ventanas, n_features)
        parts.append(arr.T.flatten())            # transpose → feature-major
        keys.append(key)

    if window_range is not None:
        start, end = window_range
        n_w = end - start + 1
        cols = [f'{feat}_w{i}' for feat in features
                for i in range(start, start + n_w)]
    else:
        cols = [f'{feat}_w{i}' for feat in features for i in range(n_windows)]

    flat = pd.DataFrame(np.vstack(parts), columns=cols)
    return pd.concat([pd.DataFrame(keys, columns=GROUPS), flat], axis=1)

In [40]:
df_flat = flatten_windows(df_pca, FEATURES) 

In [41]:
df_flat

,origin_db,subject,stimulus,VAR_STD_w0,VAR_STD_w1,VAR_STD_w2,VAR_STD_w3,VAR_STD_w4,VAR_STD_w5,VAR_STD_w6,...,SSC_w15,SSC_w16,SSC_w17,SSC_w18,SSC_w19,SSC_w20,SSC_w21,SSC_w22,SSC_w23,SSC_w24
0,NinaProDB1,1,22,-3.674658,-3.554762,-3.345881,-2.026584,-0.756450,0.800757,2.119916,...,1.907996,1.735284,1.126743,0.717940,0.240270,0.171557,-0.507768,-1.999088,-3.140061,-3.794079
1,NinaProDB1,1,25,-3.678551,-3.249742,-2.704433,-2.377162,-2.269985,-1.088096,1.588179,...,1.608417,0.233746,-1.085238,-1.780879,-2.304591,-3.345092,-3.957803,-4.639192,-4.484263,-4.499591
2,NinaProDB1,1,27,-2.535916,-2.480511,-1.637905,0.292560,0.843791,0.988654,0.438944,...,1.710213,2.124506,1.236020,0.336962,-0.751043,-1.074990,-1.287684,-2.122486,-3.491547,-4.461012
3,NinaProDB1,1,30,-3.009006,-2.823410,-1.571150,-0.902717,2.282430,3.450712,4.242549,...,1.970407,1.603166,0.875490,-0.307017,-0.737389,-0.965459,-1.913378,-2.925955,-3.827327,-3.866780
4,NinaProDB1,1,31,-2.991461,-2.860197,-2.252684,-1.818677,-0.748044,-0.065385,0.457318,...,0.612979,-0.213727,-0.253727,-2.022896,-2.462471,-3.031704,-2.760803,-3.201515,-3.592764,-4.084681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
604,NinaProDB5,10,27,-2.882137,-1.628284,-0.304534,-0.008873,0.472912,1.308857,1.954019,...,3.050308,2.972404,2.480105,0.876072,0.379155,-0.308034,-0.437723,-0.580556,-1.101005,-1.189790
605,NinaProDB5,10,30,-3.677039,-2.970050,-1.262352,-0.088434,0.988335,1.821794,2.353125,...,2.428329,2.896196,2.946686,2.257455,1.089103,-0.633673,-1.612535,-2.826666,-2.776229,-3.303582
606,NinaProDB5,10,31,-4.149849,-3.713023,-2.768422,-1.768960,-1.262061,-0.986386,-0.913301,...,1.054495,1.331010,0.272506,-0.527759,-0.114583,-0.743128,-0.824446,-1.492149,-1.454086,-1.611096
607,NinaProDB5,10,37,-1.496772,-0.144806,1.151843,1.987527,2.179634,2.359104,2.247411,...,0.218458,1.405475,1.721348,1.904045,2.128505,0.774270,0.688575,-0.471136,-0.375606,-1.170710


In [42]:
df_flat.to_csv('../data/prueba/df_flat_wnd_max.csv', index=False)

In [43]:
GROUPS = ['origin_db', 'subject', 'stimulus']
FEATURES = ['VAR_STD', 'Range', 'Energy', 'Peak_to_Peak', 'VAR', 'Max', 'MNF', 'SSC']
N_WINDOWS = 25

STATS = ['median', 'min', 'max', 'first', 'last', 'slope', 'std']


def summarize_windows(df_pca, features=FEATURES, n_windows=N_WINDOWS):
    """
    Calcula estadísticos de cada feature a lo largo de las últimas n_windows
    ventanas, para cada (origin_db, subject, stimulus).

    Estadísticos: mean, median, std, min, max, first, last, slope
    Columnas: VAR_STD_mean, VAR_STD_median, ..., VAR_STD_slope, Range_mean, ...
    slope = pendiente de regresión lineal feature ~ índice de ventana
    """
    rows, keys = [], []
    g_sorted = df_pca.sort_values('window')

    for key, g in g_sorted.groupby(GROUPS, sort=True):
        g = g.tail(n_windows)                      # últimas 25 ventanas
        arr = g[features].to_numpy()               # (n_ventanas, n_features)

        # rellenar al frente si el grupo tiene < n_windows (NaN → se ignoran)
        if arr.shape[0] < n_windows:
            pad = np.full((n_windows - arr.shape[0], len(features)), np.nan)
            arr = np.vstack([pad, arr])

        # eje x = índice de ventana 0..n_windows-1 (para slope)
        t = np.arange(arr.shape[0], dtype=float)

        stats = {}
        for j, feat in enumerate(features):
            col = arr[:, j]
            valid = np.isfinite(col)
            v, tv = col[valid], t[valid]

            stats[f'{feat}_median'] = np.nanmedian(v) if v.size else np.nan
            stats[f'{feat}_min']    = np.nanmin(v) if v.size else np.nan
            stats[f'{feat}_max']    = np.nanmax(v) if v.size else np.nan
            stats[f'{feat}_first']  = v[0] if v.size else np.nan
            stats[f'{feat}_last']   = v[-1] if v.size else np.nan
            stats[f'{feat}_std']    = np.nanstd(v) if v.size else np.nan

            # pendiente de la regresión lineal (cierre cerrado)
            if v.size >= 2 and np.ptp(tv) > 0:
                tbar, vbar = tv.mean(), v.mean()
                stats[f'{feat}_slope'] = ((tv - tbar) * (v - vbar)).sum() / \
                                         ((tv - tbar) ** 2).sum()
            else:
                stats[f'{feat}_slope'] = 0.0

        rows.append(stats)
        keys.append(key)

    cols = [f'{feat}_{s}' for feat in features for s in STATS]
    flat = pd.DataFrame(rows, columns=cols)
    keys_df = pd.DataFrame(keys, columns=GROUPS)
    return pd.concat([keys_df, flat], axis=1)

In [44]:
df_stats = summarize_windows(df_pca)               # 609 × 67
print(df_stats.shape)
df_stats.head()

(609, 59)


,origin_db,subject,stimulus,VAR_STD_median,VAR_STD_min,VAR_STD_max,VAR_STD_first,VAR_STD_last,VAR_STD_slope,VAR_STD_std,...,MNF_last,MNF_slope,MNF_std,SSC_median,SSC_min,SSC_max,SSC_first,SSC_last,SSC_slope,SSC_std
0,NinaProDB1,1,22,-0.547026,-3.674658,2.838305,-3.674658,-3.474609,0.004012,1.802954,...,0.942114,-0.031257,2.335408,0.717940,-4.526623,2.239146,-4.509600,-3.794079,0.069388,2.360612
1,NinaProDB1,1,25,-1.094770,-3.678551,3.402195,-3.678551,-3.624751,-0.090815,2.571817,...,-1.791384,-0.012453,1.588345,-1.085238,-4.639192,3.441275,-3.485096,-4.499591,-0.112049,2.850151
2,NinaProDB1,1,27,0.988654,-3.322113,2.557980,-2.535916,-3.322113,-0.029484,1.863735,...,0.415363,-0.053772,2.090755,-0.251815,-4.461012,3.328998,-3.814316,-4.461012,-0.009449,2.216125
3,NinaProDB1,1,30,2.910406,-3.597008,5.074645,-3.009006,-3.597008,-0.040799,3.054502,...,-0.831609,-0.029959,2.010770,0.875490,-4.136052,3.516397,-4.136052,-3.866780,-0.053407,2.445837
4,NinaProDB1,1,31,-1.818677,-3.588683,0.457318,-2.991461,-3.588683,-0.049609,1.111447,...,-1.618633,-0.033046,1.529390,-0.253727,-4.084681,1.223443,-3.834560,-4.084681,-0.072871,1.814540


In [45]:
df_stats.to_csv('../data/prueba/df_stats_wnd_max.csv', index=False)

In [22]:
sello.mostrar()

### Sello de procedencia

| Campo | Valor |
|---|---|
| Notebook | `00_test_datasets.ipynb` |
| Commit | `f5d3083` |
| Rama | `main` |
| Estado | sucio (12 archivos sin commitear) |
| Ejecutado | 2026-10-13T18:33:23 |

**Archivos generados en esta ejecución:**

- (ninguno detectado en `data/`, `resultados/`, `figures/`)